In [23]:
import numpy as np
import pandas as pd

In [24]:
df = pd.read_csv('../data/raw/daily-rainfall-at-state-level.csv')

In [25]:
df.head()

,id,date,state_code,state_name,actual,rfs,normal,deviation
0,0,2009-01-01,5,Uttarakhand,0.0,0.003906,2.19,-100.0
1,1,2009-01-01,18,Assam,0.0,0.000000,0.52,-100.0
2,2,2009-01-01,16,Tripura,0.0,0.000000,0.09,-100.0
3,3,2009-01-01,36,Telangana,0.0,0.000000,0.17,-100.0
4,4,2009-01-01,2,Himachal Pradesh,0.0,0.008566,3.31,-100.0


## About the columns

actual — how much rain really fell that day, in mm. Look at row 1: 0.0 — no rain that day. Row 9: 1.31 — a little over 1mm of rain fell.

normal — the "usual" amount of rain expected on that date for that state, based on many years of history.
It's not what happened this day — it's the long-term average benchmark to compare against.

deviation — how far off actual was from normal, as a percentage.
deviation tells you how "surprising" that day's rain was, relative to the season/climate for that state.

In [26]:
df.shape

(204876, 8)

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 204876 entries, 0 to 204875
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          204876 non-null  int64  
 1   date        204876 non-null  str    
 2   state_code  204876 non-null  int64  
 3   state_name  204876 non-null  str    
 4   actual      187714 non-null  float64
 5   rfs         199011 non-null  float64
 6   normal      193358 non-null  float64
 7   deviation   173855 non-null  float64
dtypes: float64(4), int64(2), str(2)
memory usage: 12.5 MB


In [28]:
df.isnull().sum()

id                0
date              0
state_code        0
state_name        0
actual        17162
rfs            5865
normal        11518
deviation     31021
dtype: int64

In [29]:
df.date.min()

'2009-01-01'

In [30]:
df.date.max()

'2024-07-31'

In [31]:
df['state_name'].unique()

<StringArray>
[                                 'Uttarakhand',
                                        'Assam',
                                      'Tripura',
                                    'Telangana',
                             'Himachal Pradesh',
                            'Jammu And Kashmir',
                                    'Meghalaya',
                                  'Lakshadweep',
                                  'Maharashtra',
                                       'Punjab',
                                    'Jharkhand',
                               'Andhra Pradesh',
                            'Arunachal Pradesh',
                                       'Ladakh',
                  'Andaman And Nicobar Islands',
                                          'Goa',
                                'Uttar Pradesh',
                                      'Haryana',
                               'Madhya Pradesh',
                                      'Manipur',
      

### Understanding the actual column

=> actual is the actual rainfall recorded that day (in mm) for a given state — this is the core signal the entire project is built on, since it will be used to define whether a day counts as "rainy," which becomes the prediction target.

In [32]:
missing_by_state = df.groupby('state_name')['actual'].apply(lambda x: x.isnull().mean())
missing_by_state = missing_by_state.sort_values(ascending=False)
print(missing_by_state.head(10))

state_name
Andaman And Nicobar Islands    1.000000
Lakshadweep                    1.000000
Arunachal Pradesh              0.029872
Andhra Pradesh                 0.029872
Assam                          0.029872
Bihar                          0.029872
Chhattisgarh                   0.029872
Chandigarh                     0.029872
Goa                            0.029872
Gujarat                        0.029872
Name: actual, dtype: float64


In [33]:
missing_dates = df[df['actual'].isnull()]['date'].value_counts()
print(missing_dates.head(10))
print('Number of distinct dates with missing actual:', missing_dates.shape[0])

date
2020-05-10    36
2020-05-11    36
2020-05-12    36
2020-07-14    36
2020-07-15    36
2020-07-16    36
2020-07-17    36
2020-07-18    36
2020-07-19    36
2020-07-20    36
Name: count, dtype: int64
Number of distinct dates with missing actual: 5691


### Checking missing values per state

actual has ~8% missing values overall, but averages can hide important patterns — a small number of badly-missing states could be pulling that number, while most states might be fine. Grouping the missing-value check by state reveals whether missingness is spread evenly or concentrated in specific states, which determines the cleaning strategy.

In [34]:
missing_dates.value_counts()

count
2     5521
36     170
Name: count, dtype: int64

### Distinguishing "always-missing states" from genuine nationwide blackout days

Two states (Andaman & Nicobar Islands, Lakshadweep) are missing on nearly every date, which inflates the missing-date count without representing a real gap. Separating out genuine blackout days (where all 36 states are missing at once) from this noise showed 170 true blackout days, clustered mostly in April–July — small enough not to significantly bias the dataset, but worth noting as a limitation.

In [35]:
blackout_dates = missing_dates[missing_dates == 36].index
pd.to_datetime(blackout_dates).month.value_counts().sort_index()

date
1     1
2     1
3    12
4    33
5    36
6    37
7    50
Name: count, dtype: int64

# Data Cleaning

### Dropping unusable states and residual missing rows

Andaman & Nicobar Islands and Lakshadweep have 100% missing rainfall data — no usable signal exists for them, so they are excluded. The remaining states have a small residual (~3%) of missing actual values, including 170 nationwide "blackout" days. Since this is a small fraction of the data and imputing rainfall values risks fabricating weather that didn't happen, these rows are dropped rather than filled in.

In [36]:
# keep only rows where the state name is NOT one of these two
df = df[(df['state_name'] != 'Andaman And Nicobar Islands') & (df['state_name'] != 'Lakshadweep')]

In [37]:
print(df.shape)
print(df['state_name'].nunique())

(193494, 8)
34


In [38]:
df = df.dropna(subset=['actual'])
print(df.shape)

(187714, 8)


After removing the two unusable states and dropping residual missing rows, the dataset shrinks from 204,876 to 187,714 rows across 34 states/UTs — a clean base to build features on.

date is currently a string, not a datetime, which prevents proper chronological sorting and date-based operations. Converting it to datetime and sorting by state_name then date ensures that when we build lag/rolling features per state later, each state's rows are in correct time order.

In [44]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['state_name', 'date']).reset_index(drop=True)
df.head()

,id,date,state_code,state_name,actual,rfs,normal,deviation
0,11,2009-01-01,28,Andhra Pradesh,0.00,0.000000,0.45,-100.0
1,46,2009-01-02,28,Andhra Pradesh,0.00,0.000000,0.53,-100.0
2,81,2009-01-03,28,Andhra Pradesh,0.02,0.097688,0.25,-92.0
3,116,2009-01-04,28,Andhra Pradesh,0.00,0.000000,0.13,-100.0
4,151,2009-01-05,28,Andhra Pradesh,0.00,0.000000,0.06,-100.0


In [46]:
df.dtypes

id                     int64
date          datetime64[us]
state_code             int64
state_name               str
actual               float64
rfs                  float64
normal               float64
deviation            float64
dtype: object

# Target Variable Definition



Going to make the new columns ("RainToday" & "RainTomorrow")

As IMD has define the threshold that if Actual-rainfall >= 2.5 then it is rainy day

| Date  | Actual-rainfall | RainToday |
|-------|-----------------|-----------|
| Jan 1 | 0.0 mm          | 0 (no rain) |
| Jan 2 | 5.0 mm          | 1 (rain) |
| Jan 3 | 0.0 mm          | 0 (no rain) |
| Jan 4 | 8.0 mm          | 1 (rain) |
| Jan 5 | 0.0 mm          | 0 (no rain) |

To predict tomorrow, each row needs to hold two things: what we know today, and what actually happened the next day (since this is historical data — we already know the outcome, we're just teaching the model the pattern).

| Date  | RainToday (what we know) | RainTomorrow (what we're trying to predict) |
|-------|--------------------------|----------------------------------------------|
| Jan 1 | 0                        | 1 ← (because Jan 2 was rain)                 |
| Jan 2 | 1                        | 0 ← (because Jan 3 was no rain)              |
| Jan 3 | 0                        | 1 ← (because Jan 4 was rain)                 |
| Jan 4 | 1                        | 0 ← (because Jan 5 was no rain)              |
| Jan 5 | 0                        | ? ← (we don't know — no Jan 6 in our data)  |

RainTomorrow value is just a copy of the next row's RainToday value, pulled backward one row. Jan 1's RainTomorrow (1) is literally Jan 2's RainToday (1). Jan 2's RainTomorrow (0) is literally Jan 3's RainToday (0). And so on.  

In [57]:
RAIN_THRESHOLD = 2.5  # mm, IMD's definition of a rainy day

df['RainToday'] = (df['actual'] >= RAIN_THRESHOLD).astype(int)
df['RainTomorrow'] = df.groupby('state_name')['RainToday'].shift(-1)

df[['date', 'state_name', 'actual', 'RainToday', 'RainTomorrow']].head(10)

,date,state_name,actual,RainToday,RainTomorrow
0,2009-01-01,Andhra Pradesh,0.00,0,0.0
1,2009-01-02,Andhra Pradesh,0.00,0,0.0
2,2009-01-03,Andhra Pradesh,0.02,0,0.0
3,2009-01-04,Andhra Pradesh,0.00,0,0.0
4,2009-01-05,Andhra Pradesh,0.00,0,0.0
5,2009-01-06,Andhra Pradesh,0.00,0,0.0
6,2009-01-07,Andhra Pradesh,0.00,0,0.0
7,2009-01-08,Andhra Pradesh,0.11,0,0.0
8,2009-01-09,Andhra Pradesh,0.17,0,0.0
9,2009-01-10,Andhra Pradesh,0.01,0,0.0


What each line does:

(df['actual'] >= RAIN_THRESHOLD) — produces True/False for every row


.astype(int) — converts True/False into 1/0, which is what scikit-learn expects for classification


df.groupby('state_name')['RainToday'].shift(-1) — for each state's group separately, moves every value up by one row, so today's row now holds tomorrow's RainToday value

In [62]:
# This te
df['RainTomorrow'].value_counts(normalize=True)

RainTomorrow
0.0    0.69829
1.0    0.30171
Name: proportion, dtype: float64

The target is moderately imbalanced (~70% no-rain, ~30% rain days) — not severe enough to require special resampling techniques, but worth keeping as a reference point: any trained model needs to clear ~70% accuracy just to beat the trivial "always predict no rain" baseline.

# Feature Engineering

##### 1) Lag features (Rain_lag1(Yesterday), Rain_lag2, Rain_lag3)

These columns tell the model whether it rained 1, 2, and 3 days ago. Rain tends to come in spells rather than isolated days, so a single day of history isn't enough — the model needs to see the recent pattern to tell the difference between an ongoing rain spell (likely to continue) and a one-off rain day breaking a dry stretch (less likely to continue). Together, these three lags give the model a short "memory" of recent rainfall to base its prediction on.


**What we want to build —** `Rain_lag1` = "what was `RainToday` **yesterday**?" for each row:

| Date  | RainToday | Rain_lag1 (yesterday's value) |
|-------|-----------|-------------------------------|
| Jan 1 | 0         | ? (no data before Jan 1)      |
| Jan 2 | 1         | 0 ← (copied from Jan 1)       |
| Jan 3 | 0         | 1 ← (copied from Jan 2)       |
| Jan 4 | 1         | 0 ← (copied from Jan 3)       |
| Jan 5 | 0         | 1 ← (copied from Jan 4)       |

In [63]:
df['Rain_lag1'] = df.groupby('state_name')['RainToday'].shift(1)
df['Rain_lag2'] = df.groupby('state_name')['RainToday'].shift(2)
df['Rain_lag3'] = df.groupby('state_name')['RainToday'].shift(3)

df[['date','state_name','RainToday','Rain_lag1','Rain_lag2','Rain_lag3']].head(10)

,date,state_name,RainToday,Rain_lag1,Rain_lag2,Rain_lag3
0,2009-01-01,Andhra Pradesh,0,NaN,NaN,NaN
1,2009-01-02,Andhra Pradesh,0,0.0,NaN,NaN
2,2009-01-03,Andhra Pradesh,0,0.0,0.0,NaN
3,2009-01-04,Andhra Pradesh,0,0.0,0.0,0.0
4,2009-01-05,Andhra Pradesh,0,0.0,0.0,0.0
5,2009-01-06,Andhra Pradesh,0,0.0,0.0,0.0
6,2009-01-07,Andhra Pradesh,0,0.0,0.0,0.0
7,2009-01-08,Andhra Pradesh,0,0.0,0.0,0.0
8,2009-01-09,Andhra Pradesh,0,0.0,0.0,0.0
9,2009-01-10,Andhra Pradesh,0,0.0,0.0,0.0


#### 2) Goal: for each day, we want to know "how much rain fell in the 3 days before this one?" — not including today.

Think about how you, a person, would guess whether it'll rain tomorrow if someone described a place to you:

"Has it been raining a lot lately?" → that's rollsum_3d / rollsum_7d

"Was it raining yesterday, or the last few days?" → that's your lag features





| Date  | Actual | rollsum_3d (sum of the 3 days BEFORE this one) |
|-------|--------|------------------------------------------------|
| Jan 1 | 2      | NaN (no days before it)                        |
| Jan 2 | 5      | NaN (only 1 day before it, not enough for a window of 3) |
| Jan 3 | 0      | NaN (only 2 days before it)                    |
| Jan 4 | 10     | 2 + 5 + 0 = **7** ← (sum of Jan 1, 2, 3)     |
| Jan 5 | 3      | 5 + 0 + 10 = **15** ← (sum of Jan 2, 3, 4)   |
| Jan 6 | 0      | 0 + 10 + 3 = **13** ← (sum of Jan 3, 4, 5)   |
| Jan 7 | 1      | 10 + 3 + 0 = **13** ← (sum of Jan 4, 5, 6)  |

In [64]:
df['rollsum_3d'] = df.groupby('state_name')['actual'].transform(lambda x: x.shift(1).rolling(3).sum())
df['rollsum_7d'] = df.groupby('state_name')['actual'].transform(lambda x: x.shift(1).rolling(7).sum())

df[['date','state_name','actual','rollsum_3d','rollsum_7d']].head(10)

,date,state_name,actual,rollsum_3d,rollsum_7d
0,2009-01-01,Andhra Pradesh,0.00,NaN,NaN
1,2009-01-02,Andhra Pradesh,0.00,NaN,NaN
2,2009-01-03,Andhra Pradesh,0.02,NaN,NaN
3,2009-01-04,Andhra Pradesh,0.00,0.02,NaN
4,2009-01-05,Andhra Pradesh,0.00,0.02,NaN
5,2009-01-06,Andhra Pradesh,0.00,0.02,NaN
6,2009-01-07,Andhra Pradesh,0.00,0.00,NaN
7,2009-01-08,Andhra Pradesh,0.11,0.00,0.02
8,2009-01-09,Andhra Pradesh,0.17,0.11,0.13
9,2009-01-10,Andhra Pradesh,0.01,0.28,0.30


#### 3) Season / month features

Rain in India follows a strong yearly pattern — the monsoon (June–September) brings much more rain than the rest of the year. Right now, the model has no way of knowing what time of year it is. Adding month and season tells it directly, so it can learn things like "rain is far more likely in July than in January."

In [66]:
df['month'] = df['date'].dt.month

def get_season(m):
    if m in [12, 1, 2]:
        return 'Winter'
    elif m in [3, 4, 5]:
        return 'Summer'
    elif m in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'PostMonsoon'

df['season'] = df['month'].apply(get_season)

df[['date','month','season']].sample(10)

,date,month,season
179071,2015-07-28,7,Monsoon
37058,2019-10-08,10,PostMonsoon
8035,2015-11-20,11,PostMonsoon
58995,2019-05-14,5,Summer
164453,2020-12-11,12,Winter
140416,2015-07-20,7,Monsoon
180559,2019-08-24,8,Monsoon
82191,2022-07-02,7,Monsoon
112967,2015-12-23,12,Winter
28141,2010-06-21,6,Monsoon


#### 4) Cleaning deviation

deviation shows how much rain differed from normal, as a percentage. But when normal is a tiny number, dividing by it can create huge, unrealistic percentages (like +50,000%). These extreme values would confuse the model, so we cap deviation between -100% and 500% — keeping the real pattern while removing the freak spikes.

In [ ]:
df['deviation_clipped'] = df['deviation'].clip(-100, 500)
# .clip(-100, 500) — any value below -100 gets set to -100, any value above 500 gets set to 500, everything in between stays untouched


df[['date','state_name','deviation','deviation_clipped']].describe()

,date,deviation,deviation_clipped
count,187714,173855.000000,173855.000000
mean,2016-07-30 04:06:28.625248,38.516222,-19.739505
min,2009-01-01 00:00:00,-100.000000,-100.000000
25%,2012-10-12 00:00:00,-100.000000,-100.000000
50%,2016-07-23 00:00:00,-89.060000,-89.060000
75%,2020-05-03 00:00:00,-9.045000,-9.045000
max,2024-03-20 00:00:00,362300.000000,500.000000
std,NaN,1367.744520,144.193110
